# PostgreSQL database (read-only)
Data dictionary, metadata, data lineage, data mapping, business glossary and data catalog for the `b2b_ecommerce` PostgreSQL database. Only SELECT statements are sent and the session is read-only. The password is read from the `PGPASSWORD` environment variable or typed into a prompt - it is never stored in this notebook.

## 0. Read-only connection and structure load

In [1]:
# =====================================================================
# POSTGRES SECTION 0 - READ-ONLY CONNECTION + CATALOG LOAD
# Everything in the PostgreSQL sections only READS from the database. Read-only is enforced three ways:
#   1) the session is opened read-only (psycopg2 readonly=True -> BEGIN READ ONLY)
#   2) the server default is switched to default_transaction_read_only=on for this connection
#   3) pg_query() refuses any statement that does not start with SELECT / WITH / SHOW / EXPLAIN
# The password is NOT written in this notebook: it is taken from the PGPASSWORD environment
# variable, or typed into a prompt when that variable is not set.
# =====================================================================
import getpass
import os
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
import psycopg2
from psycopg2 import sql

PG_SETTINGS = dict(host="localhost", port=5432, dbname="b2b_ecommerce", user="postgres")
pg_password = os.environ.get("PGPASSWORD") or getpass.getpass(f"Password for Postgres user '{PG_SETTINGS['user']}': ")

pg_conn = psycopg2.connect(
    **PG_SETTINGS,
    password=pg_password,
    connect_timeout=10,
    application_name="eda_notebook_readonly",
    options="-c default_transaction_read_only=on -c statement_timeout=60000",   # read-only + 60 s per query
)
pg_conn.set_session(readonly=True, autocommit=True)
del pg_password                                                                  # do not keep the secret around

_ALLOWED_START = ("select", "with", "show", "explain")


def pg_query(query, params=None):
    """Run ONE read-only statement and return the result as a DataFrame (statement type is checked first)."""
    text = query.as_string(pg_conn) if isinstance(query, sql.Composable) else query
    if not text.lstrip().lower().startswith(_ALLOWED_START):
        raise PermissionError(f"Only read statements are allowed here, got: {text[:40]!r}")
    with pg_conn.cursor() as cur:
        cur.execute(query, params)
        cols = [d.name for d in cur.description]
        return pd.DataFrame.from_records(cur.fetchall(), columns=cols, coerce_float=True)


def pg_scalar(query, params=None):
    """First value of the first row (e.g. a count)."""
    return pg_query(query, params).iat[0, 0]


def ident(name):
    """Safely quoted table / column name."""
    return sql.Identifier(name)


# ---- Proof that the session is read-only ------------------------------------------------------------
print("transaction_read_only :", pg_scalar("SHOW transaction_read_only"))
print("connected to          :", pg_scalar("select current_database() || ' as ' || current_user"))

# ---- Load the structure of the database (names, types, keys) --------------------------------------
pg_tables = pg_query("""
    select table_name from information_schema.tables
    where table_schema = 'public' and table_type = 'BASE TABLE' order by table_name""")["table_name"].tolist()

pg_columns = pg_query("""
    select table_name, ordinal_position as pos, column_name, data_type, character_maximum_length as char_len,
           numeric_precision as num_prec, numeric_scale as num_scale, is_nullable, column_default
    from information_schema.columns where table_schema = 'public' order by table_name, ordinal_position""")

pg_pk = pg_query("""
    select cl.relname as table_name, a.attname as column_name
    from pg_constraint c join pg_class cl on cl.oid = c.conrelid
    join lateral unnest(c.conkey) as k(attnum) on true
    join pg_attribute a on a.attrelid = c.conrelid and a.attnum = k.attnum
    where c.contype = 'p' and c.connamespace = 'public'::regnamespace""")

pg_fk = pg_query("""
    select cl.relname as child_table, a.attname as child_column, cf.relname as parent_table,
           af.attname as parent_column, c.conname as constraint_name
    from pg_constraint c
    join pg_class cl on cl.oid = c.conrelid join pg_class cf on cf.oid = c.confrelid
    join lateral unnest(c.conkey, c.confkey) as k(ck, fk) on true
    join pg_attribute a on a.attrelid = c.conrelid and a.attnum = k.ck
    join pg_attribute af on af.attrelid = c.confrelid and af.attnum = k.fk
    where c.contype = 'f' and c.connamespace = 'public'::regnamespace
    order by 1, 2""")

# exact row counts (the tables are small; pg_stat estimates are empty for the reference tables)
pg_rowcounts = {t: int(pg_scalar(sql.SQL("select count(*) from {}").format(ident(t)))) for t in pg_tables}

print(f"\nTables  : {len(pg_tables)} -> {pg_tables}")
print(f"Columns : {len(pg_columns)}   PK columns: {len(pg_pk)}   FK links: {len(pg_fk)}")
pd.Series(pg_rowcounts, name="rows").to_frame()

transaction_read_only : on
connected to          : b2b_ecommerce as postgres

Tables  : 7 -> ['companies', 'customers', 'order_items', 'orders', 'products', 'supplier_products', 'suppliers']
Columns : 87   PK columns: 8   FK links: 9


,rows
companies,425
customers,1592
order_items,29987
orders,11122
products,160
supplier_products,313
suppliers,25


## 1. Metadata
*Data about the data*: server, database, tables, sizes, keys, time coverage and volume over time.

In [2]:
# =====================================================================
# POSTGRES SECTION 1 - METADATA  ("data about the data")
# Server, database, tables, sizes, keys, time coverage and volume over time. All read-only.
# =====================================================================

# ---- 1.1 Server and database facts ---------------------------------------------------------------------
server = pg_query("""
    select version() as version,
           current_database() as database,
           pg_size_pretty(pg_database_size(current_database())) as database_size,
           pg_encoding_to_char(d.encoding) as encoding,
           d.datcollate as collation,
           current_setting('TimeZone') as timezone,
           pg_postmaster_start_time() as server_started,
           (select count(*) from pg_extension) as extensions
    from pg_database d where d.datname = current_database()""").iloc[0]

pg_facts = {
    "Database": server["database"],
    "Host : port": f"{PG_SETTINGS['host']} : {PG_SETTINGS['port']}",
    "Server version": server["version"].split(",")[0],
    "Database size": server["database_size"],
    "Encoding / collation": f"{server['encoding']} / {server['collation']}",
    "Server time zone": server["timezone"],
    "Server started": server["server_started"],
    "Schemas with user tables": ", ".join(pg_query("select distinct table_schema from information_schema.tables where table_schema not in ('pg_catalog','information_schema')")["table_schema"]),
    "Tables": len(pg_tables),
    "Views / materialized views": int(pg_scalar("select count(*) from pg_class where relnamespace = 'public'::regnamespace and relkind in ('v','m')")),
    "Total columns": len(pg_columns),
    "Total rows (all tables)": sum(pg_rowcounts.values()),
    "Session is read-only": pg_scalar("SHOW transaction_read_only"),
    "Purpose": "Operational B2B database (source 1): companies, suppliers, products, customers, orders",
    "Seeded by": "data_generation/gen_postgres.py (synthetic data), run by Airflow DAG b2b_daily_pipeline",
}
print("1.1 DATABASE FACTS")
print(pd.Series(pg_facts).to_string(), "\n")

# ---- 1.2 Table inventory (size, keys, columns) ----------------------------------------------------------
sizes = pg_query("""
    select c.relname as table_name,
           pg_total_relation_size(c.oid) as total_bytes,
           pg_relation_size(c.oid) as table_bytes,
           pg_indexes_size(c.oid) as index_bytes,
           (select count(*) from pg_index i where i.indrelid = c.oid) as indexes,
           obj_description(c.oid) as table_comment
    from pg_class c where c.relnamespace = 'public'::regnamespace and c.relkind = 'r'""").set_index("table_name")

pg_table_meta = pd.DataFrame({
    "rows": pd.Series(pg_rowcounts),
    "columns": pg_columns.groupby("table_name").size(),
    "primary_key": pg_pk.groupby("table_name")["column_name"].agg(", ".join),
    "fk_out (references)": pg_fk.groupby("child_table")["parent_table"].agg(lambda x: ", ".join(sorted(set(x)))),
    "fk_in (referenced by)": pg_fk.groupby("parent_table")["child_table"].agg(lambda x: ", ".join(sorted(set(x)))),
}).join(sizes).fillna({"fk_out (references)": "-", "fk_in (referenced by)": "-"})
pg_table_meta["total_size_kb"] = (pg_table_meta["total_bytes"] / 1024).round(0)
pg_table_meta["data_kb"] = (pg_table_meta["table_bytes"] / 1024).round(0)
pg_table_meta["index_kb"] = (pg_table_meta["index_bytes"] / 1024).round(0)
pg_table_meta["comment_in_db"] = pg_table_meta["table_comment"].fillna("(none - no COMMENT set in the database)")

print("1.2 TABLE INVENTORY")
print(pg_table_meta[["rows", "columns", "primary_key", "fk_out (references)", "fk_in (referenced by)",
                     "total_size_kb", "data_kb", "index_kb", "indexes", "comment_in_db"]].to_string(), "\n")

# ---- 1.3 Time coverage: which date/timestamp columns exist and what range they span -----------------
print("1.3 TIME COVERAGE (every date / timestamp column)")
time_cols = pg_columns[pg_columns["data_type"].str.contains("date|timestamp")][["table_name", "column_name"]]
cover = []
for t, c in time_cols.itertuples(index=False):
    r = pg_query(sql.SQL("select min({c}) as first, max({c}) as last, count({c}) as non_null from {t}")
                 .format(c=ident(c), t=ident(t))).iloc[0]
    cover.append((t, c, r["first"], r["last"], int(r["non_null"])))
pg_time_coverage = pd.DataFrame(cover, columns=["table", "column", "first", "last", "non_null"])
print(pg_time_coverage.to_string(index=False), "\n")

# ---- 1.4 How many rows were changed after they were created (change tracking evidence) -----------------
changed = []
for t in pg_tables:
    cols = set(pg_columns.loc[pg_columns["table_name"] == t, "column_name"])
    if {"created_at", "updated_at"} <= cols:
        n = int(pg_scalar(sql.SQL("select count(*) from {t} where updated_at > created_at").format(t=ident(t))))
        changed.append((t, pg_rowcounts[t], n))
print("1.4 ROWS WHERE updated_at > created_at (rows that were modified after creation)")
print(pd.DataFrame(changed, columns=["table", "rows", "modified_after_creation"]).to_string(index=False), "\n")

# ---- 1.5 Volume over time: rows per month of creation, for every table that has a created_at column ----------------
monthly = {}
for t in pg_tables:
    if "created_at" in set(pg_columns.loc[pg_columns["table_name"] == t, "column_name"]):
        m = pg_query(sql.SQL("select to_char(date_trunc('month', created_at), 'YYYY-MM') as month, count(*) as n from {t} group by 1 order by 1")
                     .format(t=ident(t))).set_index("month")["n"]
        monthly[t] = m
pg_rows_per_month = pd.DataFrame(monthly).fillna(0).astype(int)
print("1.5 ROWS PER MONTH OF created_at (tables that have the column)")
print(pg_rows_per_month.T.to_string(), "\n")

# ---- 1.6 Days covered by created_at per table ---------------------------------------------------------------------------
cov = []
for t in monthly:
    d = pg_query(sql.SQL("select min(created_at)::date as a, max(created_at)::date as b, count(distinct created_at::date) as d from {t}").format(t=ident(t))).iloc[0]
    span = (pd.Timestamp(d["b"]) - pd.Timestamp(d["a"])).days + 1
    cov.append((t, d["a"], d["b"], span, int(d["d"]), span - int(d["d"])))
print("1.6 DAY COVERAGE OF created_at")
print(pd.DataFrame(cov, columns=["table", "first_day", "last_day", "calendar_days", "days_with_rows", "days_without_rows"]).to_string(index=False))

1.1 DATABASE FACTS
Database                                                          b2b_ecommerce
Host : port                                                    localhost : 5432
Server version                                PostgreSQL 18.1 on x86_64-windows
Database size                                                             15 MB
Encoding / collation                          UTF8 / English_United States.1252
Server time zone                                                   Asia/Karachi
Server started                                 2026-10-31 11:52:41.208840+05:00
Schemas with user tables                                                 public
Tables                                                                        7
Views / materialized views                                                    0
Total columns                                                                87
Total rows (all tables)                                                   43624
Session is read-only 

## 2. Data dictionary
Every column of every table: type, keys, documented meaning and expected values, data-quality rules, and what the data really contains.

In [3]:
# =====================================================================
# POSTGRES SECTION 2 - DATA DICTIONARY
# For every column of every table: type, keys, meaning, expected values, and what the data REALLY contains
# (empties, distinct values, ranges, examples, text formats, rule violations and generic data-health findings).
# =====================================================================

# ---- 2.1 Documented meaning + expected values (gen_postgres.py DDL, common.py, docs/data_dictionary_and_kpi_mapping.md) ----
# (table, column) -> (business meaning, expected values / format)
DOC = {
    # companies
    ("companies", "company_id"): ("Unique id of the company (suppliers 1-25, buyers 26-425)", "integer"),
    ("companies", "name"): ("Legal name of the company", "unique; ends with S.A. / S.R.L. / S.A.S."),
    ("companies", "cuit"): ("Argentine tax id", "NN-NNNNNNNN-N (prefix 30/33)"),
    ("companies", "company_type"): ("Role of the company", "supplier / buyer"),
    ("companies", "country"): ("Country of the company HQ (full name, not ISO)", "Argentina, Brazil, Chile, Uruguay"),
    ("companies", "city"): ("HQ city (not a ship-to address)", "11 cities, UTF-8 accents"),
    ("companies", "rating"): ("Company rating", "2.0 - 5.0"),
    ("companies", "address"): ("Street address (noise)", "free text"),
    ("companies", "phone"): ("Company phone with dial code", "+CC ..."),
    ("companies", "contact_email"): ("Company-level e-mail (not the customer's e-mail)", "info@domain.com"),
    ("companies", "industry"): ("Industry sector", "9 industries"),
    ("companies", "employee_count"): ("Head count (correlates with order volume)", "positive integer"),
    ("companies", "payment_terms"): ("Agreed payment terms", "Net 15 / Net 30 / Net 45 / Prepaid"),
    ("companies", "is_active"): ("Company active flag", "true / false"),
    ("companies", "created_at"): ("Onboarding timestamp; buyers cannot order before it", "timestamp"),
    ("companies", "updated_at"): ("Last change timestamp (CDC watermark)", ">= created_at"),
    # products
    ("products", "product_id"): ("Unique id of the product", "1-160"),
    ("products", "sku"): ("Stock keeping unit; also appears in web-log URLs", "CAT-NNNN, e.g. ELE-0001"),
    ("products", "name"): ("Product name", "e.g. Pro Wireless Mouse"),
    ("products", "category"): ("Product category", "8 categories"),
    ("products", "list_price"): ("Catalog selling price", "positive"),
    ("products", "brand"): ("Brand (random, unrelated to category)", "7 brands"),
    ("products", "unit_of_measure"): ("Unit the product is sold in", "unit / box / pack / set"),
    ("products", "weight_kg"): ("Weight (noise)", "positive"),
    ("products", "barcode"): ("13-digit barcode (noise)", "prefix 779"),
    ("products", "is_active"): ("Active in catalog (inactive products still sell)", "true / false"),
    ("products", "min_order_qty"): ("Minimum order quantity (not enforced)", "1 / 5 / 10"),
    ("products", "created_at"): ("Product creation timestamp", "timestamp"),
    ("products", "updated_at"): ("Last change timestamp", ">= created_at"),
    # suppliers
    ("suppliers", "supplier_id"): ("Supplier id (= company_id of the supplier)", "1-25"),
    ("suppliers", "company_id"): ("Company row that holds the supplier's name / country / city", "FK companies"),
    ("suppliers", "lead_time_days"): ("Supplier-level delivery lead time", "2 - 30 days"),
    ("suppliers", "contact_name"): ("Sales contact name", "text"),
    ("suppliers", "contact_email"): ("Sales contact e-mail", "sales@domain.com"),
    ("suppliers", "min_order_value"): ("Minimum order value (not enforced)", "0 / 500 / 1000 / 2500"),
    ("suppliers", "is_preferred"): ("Preferred supplier flag (independent of supplier_products.is_preferred)", "true / false"),
    ("suppliers", "created_at"): ("= company created_at", "timestamp"),
    ("suppliers", "updated_at"): ("Last change timestamp", ">= created_at"),
    # supplier_products
    ("supplier_products", "supplier_id"): ("Supplier of the pair", "FK suppliers"),
    ("supplier_products", "product_id"): ("Product of the pair", "FK products"),
    ("supplier_products", "supplier_price"): ("Supplier's price = cost of goods proxy (single static price)", "positive"),
    ("supplier_products", "supplier_sku"): ("Supplier's own code", "S<supplier>-<sku>"),
    ("supplier_products", "min_order_qty"): ("Pair-level minimum quantity (not enforced)", "1 / 5 / 10 / 25"),
    ("supplier_products", "lead_time_days"): ("Pair-level lead time (differs from suppliers.lead_time_days)", "2 - 30 days"),
    ("supplier_products", "is_preferred"): ("Preferred for this product", "true / false"),
    ("supplier_products", "valid_from"): ("Date the price is valid from (= product created date)", "date"),
    ("supplier_products", "created_at"): ("Creation timestamp", "timestamp"),
    ("supplier_products", "updated_at"): ("Last change timestamp", ">= created_at"),
    # customers
    ("customers", "customer_id"): ("Unique id of the person who orders", "integer"),
    ("customers", "company_id"): ("Employer (buyer company)", "FK companies"),
    ("customers", "birth_date"): ("Date of birth (noise)", "1960 - 2000"),
    ("customers", "gender"): ("Gender", "M / F"),
    ("customers", "first_name"): ("First name", "text"),
    ("customers", "last_name"): ("Last name", "text"),
    ("customers", "email"): ("Login e-mail = cross-source key to web logs and Won leads", "lower case first.last<id>@domain.com"),
    ("customers", "phone"): ("Phone with dial code", "+CC [9] AA NNNN-NNNN"),
    ("customers", "job_title"): ("Job title", "6 titles"),
    ("customers", "is_active"): ("Active flag (ignored by the order generator)", "true / false"),
    ("customers", "document_number"): ("National id (DNI-like)", "8 digits"),
    ("customers", "created_at"): ("= buyer company created_at", "timestamp"),
    ("customers", "updated_at"): ("Last change timestamp", ">= created_at"),
    # orders
    ("orders", "order_id"): ("Order id; encodes the creation date as YYYYMMDD*1000 + sequence", "bigint"),
    ("orders", "buyer_company_id"): ("Company that buys", "FK companies"),
    ("orders", "customer_id"): ("Person who placed the order", "FK customers"),
    ("orders", "order_date"): ("Business date of the order", "= date(created_at), not in the future"),
    ("orders", "status"): ("Order lifecycle stage; changes in place", "pending > confirmed > shipped > delivered, or cancelled (lower case)"),
    ("orders", "payment_method"): ("How the buyer pays", "bank_transfer / credit_card / cheque / cash_on_delivery"),
    ("orders", "shipping_method"): ("Delivery type", "standard / express / pickup"),
    ("orders", "shipping_cost"): ("Order-level shipping cost (not part of revenue)", ">= 0 (0 for pickup)"),
    ("orders", "po_number"): ("Buyer's purchase-order number (20% legitimately empty)", "text"),
    ("orders", "expected_delivery_date"): ("Promised delivery date", ">= order_date"),
    ("orders", "channel"): ("Where the order came from (only 'web' has a web-log footprint)", "web / sales_rep / email / phone"),
    ("orders", "created_at"): ("Exact order time - the reliable timestamp, never corrupted", "timestamp"),
    ("orders", "updated_at"): ("Last change (status moves bump it)", ">= created_at"),
    # order_items
    ("order_items", "order_item_id"): ("Line id = order_id*10 + line_number", "bigint"),
    ("order_items", "order_id"): ("Parent order", "FK orders"),
    ("order_items", "product_id"): ("Product sold", "FK products"),
    ("order_items", "supplier_id"): ("Supplier actually used for the line", "FK suppliers; pair exists in supplier_products"),
    ("order_items", "quantity"): ("Units sold", "> 0"),
    ("order_items", "unit_price"): ("Price actually charged per unit (already net of discount)", "> 0"),
    ("order_items", "line_number"): ("Position of the line inside the order", "1..n"),
    ("order_items", "discount_pct"): ("Informational discount; already inside unit_price", "0 - 20"),
    ("order_items", "tax_rate"): ("Tax rate; not included in line_total", "21.0 or 10.5"),
    ("order_items", "line_total"): ("quantity x unit_price calculated before corruption (reliable revenue field)", "= quantity x unit_price"),
    ("order_items", "warehouse"): ("Dispatch warehouse (random, unrelated to geography)", "BsAs-Central / Cordoba-Norte / Rosario-Sur"),
    ("order_items", "created_at"): ("Creation timestamp", "timestamp"),
    ("order_items", "updated_at"): ("Last change timestamp", ">= created_at"),
}

# ---- 2.2 Data-quality rules: each is a WHERE clause counted with SELECT count(*) (read-only) ---------------
# (table, column, what the rule looks for, WHERE clause)
RULES = [
    ("companies", "cuit", "missing", "cuit is null"),
    ("companies", "cuit", "not NN-NNNNNNNN-N (incl. natural no-hyphen form)", "cuit is not null and cuit !~ '^\\d{2}-\\d{8}-\\d$'"),
    ("companies", "cuit", "broken (not even 11 digits with optional hyphens)", "cuit is not null and cuit !~ '^\\d{2}-?\\d{8}-?\\d$'"),
    ("companies", "rating", "missing", "rating is null"),
    ("companies", "phone", "missing", "phone is null"),
    ("companies", "country", "not one of the 4 expected names", "country is null or country not in ('Argentina','Brazil','Chile','Uruguay')"),
    ("companies", "company_type", "not supplier / buyer", "company_type is null or company_type not in ('supplier','buyer')"),
    ("products", "sku", "not CAT-NNNN", "sku is null or sku !~ '^[A-Z]{3}-\\d{4}$'"),
    ("products", "list_price", "missing or <= 0", "list_price is null or list_price <= 0"),
    ("supplier_products", "supplier_price", "missing or <= 0", "supplier_price is null or supplier_price <= 0"),
    ("customers", "gender", "not M / F (female, Male, empty, NULL ...)", "gender is null or gender not in ('M','F')"),
    ("customers", "email", "upper case or padded with spaces", "email <> lower(btrim(email))"),
    ("customers", "email", "missing", "email is null"),
    ("customers", "phone", "missing", "phone is null"),
    ("customers", "document_number", "missing or empty", "document_number is null or btrim(document_number) = ''"),
    ("customers", "document_number", "not 8 digits", "document_number is not null and btrim(document_number) <> '' and document_number !~ '^\\d{8}$'"),
    ("orders", "order_date", "missing", "order_date is null"),
    ("orders", "order_date", "in the future", "order_date > current_date"),
    ("orders", "order_date", "different from date(created_at)", "order_date is not null and order_date <> created_at::date"),
    ("orders", "status", "not clean lower-case lifecycle value", "status is null or status not in ('pending','confirmed','shipped','delivered','cancelled')"),
    ("orders", "payment_method", "not one of the 4 methods", "payment_method is null or payment_method not in ('bank_transfer','credit_card','cheque','cash_on_delivery')"),
    ("orders", "shipping_cost", "negative", "shipping_cost < 0"),
    ("orders", "expected_delivery_date", "before order_date", "expected_delivery_date < order_date"),
    ("orders", "channel", "not one of the 4 channels", "channel is null or channel not in ('web','sales_rep','email','phone')"),
    ("order_items", "quantity", "zero, negative or missing", "quantity is null or quantity <= 0"),
    ("order_items", "unit_price", "missing, zero or negative", "unit_price is null or unit_price <= 0"),
    ("order_items", "line_total", "different from quantity x unit_price", "line_total is distinct from quantity * unit_price"),
    ("order_items", "order_id", "duplicate line (same order + product again)",
     "order_item_id in (select order_item_id from (select order_item_id, row_number() over (partition by order_id, product_id order by order_item_id) rn from order_items) x where rn > 1)"),
    ("order_items", "discount_pct", "negative", "discount_pct < 0"),
    ("order_items", "tax_rate", "not 21.0 / 10.5", "tax_rate is null or tax_rate not in (21.0, 10.5)"),
]
# every table: update before creation is impossible
for _t in pg_tables:
    RULES.append((_t, "updated_at", "earlier than created_at", "updated_at < created_at"))

rule_rows = []
for t, c, what, where in RULES:
    n = int(pg_scalar(sql.SQL("select count(*) from {t} where ").format(t=ident(t)) + sql.SQL(where)))
    rule_rows.append((t, c, what, n, round(n / pg_rowcounts[t] * 100, 2)))
pg_rule_results = pd.DataFrame(rule_rows, columns=["table", "column", "rule (what is wrong)", "rows_affected", "rows_%"])


NUMERIC = ("integer", "bigint", "smallint", "numeric", "real", "double precision")
DATETIME = ("date", "timestamp without time zone", "timestamp with time zone")

# comments stored in the database for columns (COMMENT ON COLUMN ...)
col_comments = pg_query("""
    select c.relname as table_name, a.attname as column_name, d.description as comment_in_db
    from pg_description d join pg_class c on c.oid = d.objoid and c.relnamespace = 'public'::regnamespace
    join pg_attribute a on a.attrelid = c.oid and a.attnum = d.objsubid where d.objsubid > 0""")
comment_lookup = {(r.table_name, r.column_name): r.comment_in_db for r in col_comments.itertuples()}

# ---- 2.3 Profile every column: ONE aggregate query per table -----------------------------------------------------
profile_rows = []
for t in pg_tables:
    cols = pg_columns[pg_columns["table_name"] == t]
    parts = []
    for c, dt in zip(cols["column_name"], cols["data_type"]):
        i = ident(c)
        add = lambda expr, alias: parts.append(sql.SQL(expr).format(i=i, a=ident(f"{c}__{alias}")))
        add("count({i}) as {a}", "nn")
        add("count(distinct {i}) as {a}", "dist")
        if dt == "boolean":
            add("count(*) filter (where {i}) as {a}", "true")
        else:
            add("min({i})::text as {a}", "min")
            add("max({i})::text as {a}", "max")
        if dt in NUMERIC:
            add("avg({i})::numeric(18,2)::text as {a}", "avg")
            add("count(*) filter (where {i} < 0) as {a}", "neg")
            add("count(*) filter (where {i} = 0) as {a}", "zero")
        if dt in DATETIME:
            add("count(*) filter (where {i} > now()) as {a}", "future")
        if dt == "text":
            add("count(*) filter (where {i} <> btrim({i})) as {a}", "pad")
            add("count(*) filter (where btrim({i}) = '') as {a}", "empty")
            add("count(distinct {i}) - count(distinct lower(btrim({i}))) as {a}", "case")
    stats = pg_query(sql.SQL("select ") + sql.SQL(", ").join(parts) + sql.SQL(" from {t}").format(t=ident(t))).iloc[0]

    for c, dt in zip(cols["column_name"], cols["data_type"]):
        nn, dist = int(stats[f"{c}__nn"]), int(stats[f"{c}__dist"])
        rows_t = pg_rowcounts[t]
        g = lambda k: int(stats[f"{c}__{k}"]) if f"{c}__{k}" in stats else 0
        rng = (f"true: {g('true')} / false: {nn - g('true')}" if dt == "boolean" else f"{stats[f'{c}__min']} .. {stats[f'{c}__max']}")

        allowed, formats = "-", "-"
        if dist <= 12 and nn:                                           # low-cardinality: list every value
            vc = pg_query(sql.SQL("select {i}::text as v, count(*) as n from {t} where {i} is not null group by 1 order by 2 desc")
                          .format(i=ident(c), t=ident(t)))
            allowed = dict(zip(vc["v"], vc["n"]))
        elif dt == "text" and nn:                                       # text formats: letters -> a, digits -> 9
            m = pg_query(sql.SQL("""select mask, count(*) as n from (
                         select regexp_replace(regexp_replace({i}, '[A-Za-z]+', 'a', 'g'), '[0-9]+', '9', 'g') as mask
                         from {t} where {i} is not null) x group by 1 order by 2 desc""").format(i=ident(c), t=ident(t)))
            formats = f"{len(m)} format(s); top: " + ", ".join(f"'{a}' x{b}" for a, b in zip(m['mask'].head(3), m['n'].head(3)))
        ex = pg_query(sql.SQL("select distinct {i}::text as v from {t} where {i} is not null limit 3").format(i=ident(c), t=ident(t)))["v"].tolist()

        issues = []
        if rows_t - nn: issues.append(f"{rows_t - nn} empty (NULL)")
        if g("empty"): issues.append(f"{g('empty')} blank text")
        if g("pad"): issues.append(f"{g('pad')} padded with spaces")
        if g("case"): issues.append(f"{g('case')} spellings differ only by case/spaces")
        if g("neg"): issues.append(f"{g('neg')} negative")
        if g("zero"): issues.append(f"{g('zero')} zero")
        if g("future"): issues.append(f"{g('future')} in the future")
        profile_rows.append({
            "table": t, "column": c, "non_null": nn, "null": rows_t - nn,
            "null_%": round((rows_t - nn) / rows_t * 100, 2) if rows_t else 0.0,
            "distinct": dist, "min..max": rng, "mean": stats.get(f"{c}__avg", "-"),
            "negative": g("neg"), "zero": g("zero"), "future": g("future"), "padded": g("pad"),
            "blank_text": g("empty"), "case_variants": g("case"),
            "allowed_values_found": allowed, "text_formats": formats, "examples": ex,
            "findings": "; ".join(issues) if issues else "-",
            "finding_count": (rows_t - nn) and 1 or 0,
        })
pg_profile = pd.DataFrame(profile_rows)

# ---- 2.4 Assemble the dictionary ------------------------------------------------------------------------------------
pk_set = {(r.table_name, r.column_name) for r in pg_pk.itertuples()}
fk_map = {(r.child_table, r.child_column): f"{r.parent_table}.{r.parent_column}" for r in pg_fk.itertuples()}
unique_cols = {(r.table_name, r.column_name) for r in pg_query("""
    select cl.relname as table_name, a.attname as column_name
    from pg_index i join pg_class cl on cl.oid = i.indrelid join pg_attribute a on a.attrelid = i.indrelid and a.attnum = any(i.indkey)
    where i.indisunique and cl.relnamespace = 'public'::regnamespace""").itertuples()}


def key_label(t, c):
    parts = []
    if (t, c) in pk_set: parts.append("PK")
    if (t, c) in fk_map: parts.append(f"FK -> {fk_map[(t, c)]}")
    return " ".join(parts) or "-"


def role_of(t, c, dt, dist):
    """Kind of field, decided only from type, keys and observed cardinality."""
    if (t, c) in pk_set or (t, c) in fk_map: return "identifier"
    if dt == "boolean": return "flag"
    if dt in DATETIME: return "date/time"
    if dt in NUMERIC: return "number"
    return "category" if dist <= 12 else "free text / code"


pg_data_dictionary = pg_columns.rename(columns={"table_name": "table", "column_name": "column"}).merge(pg_profile, on=["table", "column"])
pg_data_dictionary["type"] = pg_data_dictionary.apply(
    lambda r: r["data_type"] + (f"({int(r['num_prec'])},{int(r['num_scale'])})" if r["data_type"] == "numeric" and pd.notna(r["num_prec"]) else
                                f"({int(r['char_len'])})" if pd.notna(r["char_len"]) else ""), axis=1)
pg_data_dictionary["key"] = [key_label(t, c) for t, c in zip(pg_data_dictionary["table"], pg_data_dictionary["column"])]
pg_data_dictionary["role"] = [role_of(t, c, dt, d) for t, c, dt, d in zip(pg_data_dictionary["table"], pg_data_dictionary["column"], pg_data_dictionary["data_type"], pg_data_dictionary["distinct"])]
pg_data_dictionary["nullable_in_ddl"] = pg_data_dictionary["is_nullable"]
pg_data_dictionary["default"] = pg_data_dictionary["column_default"].fillna("-")
pg_data_dictionary["unique_index"] = [(t, c) in unique_cols for t, c in zip(pg_data_dictionary["table"], pg_data_dictionary["column"])]
pg_data_dictionary["comment_in_db"] = [comment_lookup.get((t, c), "(no description stored in the database)") for t, c in zip(pg_data_dictionary["table"], pg_data_dictionary["column"])]
pg_data_dictionary["meaning"] = [DOC.get((t, c), ("(not documented)", "?"))[0] for t, c in zip(pg_data_dictionary["table"], pg_data_dictionary["column"])]
pg_data_dictionary["expected"] = [DOC.get((t, c), ("(not documented)", "?"))[1] for t, c in zip(pg_data_dictionary["table"], pg_data_dictionary["column"])]
rules_by_col = pg_rule_results[pg_rule_results["rows_affected"] > 0].groupby(["table", "column"])
# rule_violations = sum of the rows hit by every rule on that column (rules can overlap, so it is a "how dirty" score, not a distinct row count)
pg_data_dictionary["rule_violations"] = [int(rules_by_col.get_group((t, c))["rows_affected"].sum()) if (t, c) in rules_by_col.groups else 0
                                         for t, c in zip(pg_data_dictionary["table"], pg_data_dictionary["column"])]
pg_data_dictionary["rules_triggered"] = [" | ".join(f"{r['rule (what is wrong)']}: {r['rows_affected']}" for _, r in rules_by_col.get_group((t, c)).iterrows())
                                         if (t, c) in rules_by_col.groups else "-" for t, c in zip(pg_data_dictionary["table"], pg_data_dictionary["column"])]
pg_data_dictionary = pg_data_dictionary.drop(columns=["finding_count"]).set_index(["table", "pos"]).sort_index()

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)
pg_data_dictionary[["column", "type", "role", "key", "nullable_in_ddl", "meaning", "expected", "default", "unique_index", "comment_in_db"]]

column                         type              role                             key nullable_in_ddl                                                                meaning  \
table             pos                                                                                                                                                                                                 
companies         1                company_id                      integer        identifier                              PK              NO               Unique id of the company (suppliers 1-25, buyers 26-425)   
                  2                      name                         text  free text / code                               -             YES                                              Legal name of the company   
                  3                      cuit                         text  free text / code                               -             YES                                                       Argentine tax id   
                  4              company_type                         text          category                               -             YES                                                    Role of the company   
                  5                   country                         text          category                               -             YES                         Country of the company HQ (full name, not ISO)   
                  6                      city                         text          category                               -             YES                                        HQ city (not a ship-to address)   
                  7                    rating                 numeric(2,1)            number                               -             YES                                                         Company rating   
                  8                   address                         text  free text / code                               -             YES                                                 Street address (noise)   
                  9                     phone                         text  free text / code                               -             YES                                           Company phone with dial code   
                  10            contact_email                         text  free text / code                               -             YES                       Company-level e-mail (not the customer's e-mail)   
                  11                 industry                         text          category                               -             YES                                                        Industry sector   
                  12           employee_count                      integer            number                               -             YES                              Head count (correlates with order volume)   
                  13            payment_terms                         text          category                               -             YES                                                   Agreed payment terms   
                  14                is_active                      boolean              flag                               -             YES                                                    Company active flag   
                  15               created_at  timestamp without time zone         date/time                               -             YES                    Onboarding timestamp; buyers cannot order before it   
                  16               updated_at  timestamp without time zone         date/time                               -             YES                                  Last change timestamp (CDC watermark)   
customers         1               customer_id                      integer        identifier                              PK              NO                             

In [4]:
# ---- 2.3 Second view: what the data really contains (empties, distinct values, range, mean) -----------------------
pg_data_dictionary[["column", "non_null", "null", "null_%", "distinct", "min..max", "mean", "negative", "zero", "future", "rule_violations"]]

column  non_null  null  null_%  distinct                                                               min..max             mean  negative  zero  future  rule_violations
table             pos                                                                                                                                                                                           
companies         1                company_id       425     0    0.00       425                                                               1 .. 425           213.00         0     0       0                0
                  2                      name       425     0    0.00       425                        Andes Alimentos S.A.S. .. Zenith Trading S.R.L.                -         0     0       0                0
                  3                      cuit       417     8    1.88       393                                                   00-00000000-0 .. N/A                -         0     0       0               83
                  4              company_type       425     0    0.00         2                                                      buyer .. supplier                -         0     0       0                0
                  5                   country       425     0    0.00         4                                                   Argentina .. Uruguay                -         0     0       0                0
                  6                      city       425     0    0.00        11                                             Buenos Aires .. Valparaíso                -         0     0       0                0
                  7                    rating       408    17    4.00        31                                                             2.0 .. 5.0             3.49         0     0       0               17
                  8                   address       425     0    0.00       424                             Av. 9 de Julio 1052 .. Calle Sarmiento 930                -         0     0       0                0
                  9                     phone       410    15    3.53       410                                  +54 10 5019-8760 .. +598 92 7385-7199                -         0     0       0               15
                  10            contact_email       425     0    0.00       252                      info@andesalimentos.com .. info@zenithtrading.com                -         0     0       0                0
                  11                 industry       425     0    0.00         9                                              Agriculture .. Technology                -         0     0       0                0
                  12           employee_count       425     0    0.00       115                                                               3 .. 533            45.77         0     0       0                0
                  13            payment_terms       425     0    0.00         4                                                      Net 15 .. Prepaid                -         0     0       0                0
                  14                is_active       425     0    0.00         2                                                  true: 408 / false: 17                -         0     0       0                0
                  15               created_at       425     0    0.00       425                             2022-05-26 01:35:47 .. 2026-09-29 09:22:23                -         0     0       0                0
                  16               updated_at       425     0    0.00       425                             2022-05-26 01:35:47 .. 2026-09-29 09:22:23                -         0     0       0                0
customers         1               customer_id      1592     0    0.00      1592                                                              1 .. 1592           796.50         0     0       0                0
                  2                company_id      1592     0    0.0

In [ ]:
# ---- 2.4 Third view: every value of low-cardinality columns, text formats and examples -----------------------------
pg_data_dictionary[["column", "allowed_values_found", "text_formats", "examples", "rules_triggered"]]

In [ ]:
# ---- 2.5 Data-quality rules that found problems (each rule from 2.2, measured in the database) -----------------------
print("2.5 RULES THAT FOUND SOMETHING")
display(pg_rule_results[pg_rule_results["rows_affected"] > 0].reset_index(drop=True))
print(f"Rules checked: {len(pg_rule_results)}   | rules with findings: {int((pg_rule_results['rows_affected'] > 0).sum())}")

# ---- 2.6 Data-health findings found by generic checks (no business rules are assumed) --------------------------------
# Checks: empty values, blank text, text padded with spaces, spellings that differ only by case/spaces,
# negative / zero numbers, dates in the future.
print("2.6 COLUMNS WITH FINDINGS")
display(pg_data_dictionary[pg_data_dictionary["findings"] != "-"].reset_index()[["table", "column", "type", "findings"]])

# ---- 2.7 Order between date / timestamp columns of the same table ------------------------------------------------------
# For every pair of date/timestamp columns in a table: how many rows have a < b, a = b, a > b.
# A pair that is never 'a < b' tells you b is always on or before a (an ordering rule the data follows).
rows = []
for t in pg_tables:
    dcols = pg_columns[(pg_columns["table_name"] == t) & (pg_columns["data_type"].isin(DATETIME))]["column_name"].tolist()
    for a in dcols:
        for b in dcols:
            if a >= b:
                continue
            r = pg_query(sql.SQL("""select count(*) filter (where {a} < {b}) as a_before_b,
                                           count(*) filter (where {a} = {b}) as equal,
                                           count(*) filter (where {a} > {b}) as a_after_b,
                                           count(*) as comparable
                                    from {t} where {a} is not null and {b} is not null""").format(a=ident(a), b=ident(b), t=ident(t))).iloc[0]
            rule = ("always equal" if r["a_before_b"] == 0 and r["a_after_b"] == 0 else
                    "always a >= b" if r["a_before_b"] == 0 else "always a <= b" if r["a_after_b"] == 0 else "mixed")
            rows.append((t, a, b, int(r["a_before_b"]), int(r["equal"]), int(r["a_after_b"]), int(r["comparable"]), rule))
pg_date_order = pd.DataFrame(rows, columns=["table", "column_a", "column_b", "a_before_b", "a_equals_b", "a_after_b", "rows_compared", "observed_order"])
print("2.7 ORDER BETWEEN DATE COLUMNS (per table)")
display(pg_date_order)

## 3. Data lineage
End-to-end lineage stages, column-level derivation rules, change history, and what the database itself shows (dependencies, write activity, derived columns).

In [ ]:
# =====================================================================
# POSTGRES SECTION 3 - DATA LINEAGE  (where the data is born, how it moves, how it changes, where it goes)
# Stages come from the project code (common.py, gen_postgres.py, airflow2/dags/b2b_daily_pipeline.py, requirements.md).
# "evidence_in_db" is measured from the live database (read-only).
# =====================================================================

# ---- 3.1 End-to-end lineage ----------------------------------------------------------------------------------
n_modified = int(pg_scalar("select count(*) from orders where updated_at > created_at"))
last_change = pg_scalar("select max(updated_at) from orders")

pg_lineage_stages = pd.DataFrame([
    (1, "common.build_reference() / build_orders(day)", "Fixed random seed + business rules",
        "Creates the clean reference data (companies, suppliers, products, supplier_products, customers) and the orders + lines of each day; "
        "HARD_MODE injects wrong values and records every one", "Clean + dirty rows in memory", "implemented", ""),
    (2, "gen_postgres.py", "Rows from stage 1",
        "Creates the 7 tables (DDL), loads reference tables once, inserts orders / lines per day with ON CONFLICT DO NOTHING. "
        "Daily runs also UPDATE the status of ~40 open orders and move updated_at (simulates source changes)",
        "PostgreSQL database b2b_ecommerce", "implemented",
        f"{sum(pg_rowcounts.values()):,} rows in {len(pg_tables)} tables; {n_modified:,} orders modified after creation (latest change {last_change})"),
    (3, "Airflow DAG b2b_daily_pipeline - task sync_to_neon", "This PostgreSQL (connection postgres_default)",
        "Re-creates the same DDL in Neon, then upserts every table in FK order by primary key; "
        "only rows with updated_at >= (Neon watermark - 3 days), or everything on a backfill",
        "Neon PostgreSQL (copy)", "implemented", "Neon is not reachable from here - cannot be checked"),
    (4, "Databricks JDBC / Lakehouse Federation -> stg_pg_* tables", "PostgreSQL / Neon",
        "Planned: raw staging copies with CDC watermark (orders, order_items use a rolling window)",
        "Staging Delta tables (Unity Catalog)", "planned (requirements.md)", ""),
    (5, "Warehouse transforms (MERGE, star schema)", "Staging tables",
        "Planned: clean values, dedupe lines, build dim_company (SCD2), dim_customer, dim_product, dim_supplier and fact_order_item",
        "Star schema Delta tables", "planned (requirements.md)", ""),
    (6, "KPI views", "Warehouse + web logs + leads",
        "Planned: revenue, margin, CLV, supplier KPIs (K1-K10, K13, K14)", "Business reports", "planned (requirements.md)", ""),
    (7, "THIS NOTEBOOK", "Live PostgreSQL (read-only session)", "Read-only profiling and documentation", "DataFrames in memory", "implemented",
        f"connected as {PG_SETTINGS['user']}@{PG_SETTINGS['host']}:{PG_SETTINGS['port']}/{PG_SETTINGS['dbname']}"),
], columns=["stage", "component", "input", "transformation", "output", "status", "evidence_in_db"]).set_index("stage")

print("3.1 END-TO-END LINEAGE")
print("common.py generators -> gen_postgres.py -> PostgreSQL (this DB) -> Airflow sync_to_neon -> Neon -> (planned) staging -> warehouse -> KPI views\n")
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 250)
pg_lineage_stages

In [ ]:
# ---- 3.2 Column-level lineage: values that are DERIVED from other values, tested against the live data --------------
# For each derivation rule documented by the generator, the SQL counts how many rows follow it.
DERIVATIONS = [
    # (table.column, derived from, rule, SQL: rows that follow the rule, SQL: rows that can be tested)
    ("orders.order_id", "orders.created_at", "YYYYMMDD x 1000 + sequence (id holds the creation date)",
     "select count(*) from orders where order_id / 1000 = to_char(created_at, 'YYYYMMDD')::bigint",
     "select count(*) from orders"),
    ("orders.order_date", "orders.created_at", "order_date = date(created_at)",
     "select count(*) from orders where order_date = created_at::date", "select count(*) from orders where order_date is not null"),
    ("order_items.order_item_id", "order_items.order_id, line_number", "order_id x 10 + line_number",
     "select count(*) from order_items where order_item_id = order_id * 10 + line_number", "select count(*) from order_items"),
    ("order_items.line_total", "quantity, unit_price", "quantity x unit_price (calculated BEFORE corruption)",
     "select count(*) from order_items where line_total = quantity * unit_price", "select count(*) from order_items"),
    ("order_items.supplier_id", "supplier_products", "(supplier_id, product_id) must exist as a supplier_products pair",
     "select count(*) from order_items i join supplier_products sp on sp.supplier_id = i.supplier_id and sp.product_id = i.product_id",
     "select count(*) from order_items"),
    ("suppliers.supplier_id", "companies.company_id", "supplier_id = company_id of the supplier company",
     "select count(*) from suppliers where supplier_id = company_id", "select count(*) from suppliers"),
    ("customers.created_at", "companies.created_at", "customer created_at = employer company created_at",
     "select count(*) from customers c join companies co on co.company_id = c.company_id where c.created_at = co.created_at",
     "select count(*) from customers"),
    ("suppliers.created_at", "companies.created_at", "supplier created_at = its company created_at",
     "select count(*) from suppliers s join companies co on co.company_id = s.company_id where s.created_at = co.created_at",
     "select count(*) from suppliers"),
    ("supplier_products.valid_from", "products.created_at", "valid_from = date(product created_at)",
     "select count(*) from supplier_products sp join products p on p.product_id = sp.product_id where sp.valid_from = p.created_at::date",
     "select count(*) from supplier_products"),
    ("orders.customer_id", "customers.company_id", "the ordering customer works at the buying company",
     "select count(*) from orders o join customers c on c.customer_id = o.customer_id where c.company_id = o.buyer_company_id",
     "select count(*) from orders"),
    ("orders.created_at", "companies.created_at", "buyer cannot order before it was onboarded",
     "select count(*) from orders o join companies c on c.company_id = o.buyer_company_id where o.created_at >= c.created_at",
     "select count(*) from orders"),
]
rows = []
for target, source, rule, ok_sql, all_sql in DERIVATIONS:
    ok, total = int(pg_scalar(ok_sql)), int(pg_scalar(all_sql))
    rows.append((target, source, rule, total, ok, total - ok, f"{ok / total * 100:.2f}%" if total else "n/a"))
pg_column_lineage = pd.DataFrame(rows, columns=["derived_column", "derived_from", "rule", "rows_tested", "follow_rule", "break_rule", "follow_%"])

print("3.2 COLUMN-LEVEL LINEAGE (derivation rules tested in the database)")
display(pg_column_lineage)

# ---- 3.3 Change lineage: which rows changed after being created, and when -------------------------------------------
print("3.3 CHANGE LINEAGE - orders modified after creation, by the day of the change (latest 15 days)")
display(pg_query("""
    select updated_at::date as change_day, count(*) as orders_changed,
           count(*) filter (where lower(btrim(status)) = 'delivered') as now_delivered,
           count(*) filter (where lower(btrim(status)) = 'shipped') as now_shipped,
           count(*) filter (where lower(btrim(status)) = 'confirmed') as now_confirmed
    from orders where updated_at > created_at + interval '1 minute'
    group by 1 order by 1 desc limit 15"""))

In [ ]:
# =====================================================================
# POSTGRES SECTION 3 - DATA LINEAGE  (only what the database itself can show)
# A database does not record where its data came from or where it goes next, so upstream sources and downstream
# consumers are NOT listed here. What it does show: how tables depend on each other, whether any object transforms
# data inside the database, how rows were written / changed over time, and which values are derived from other columns.
# =====================================================================

# ---- 3.4 Table-level lineage from the foreign keys (parents feed children) and the load order they imply ------------------
depends_on = {t: set() for t in pg_tables}
for r in pg_fk.itertuples():
    if r.child_table != r.parent_table:
        depends_on[r.child_table].add(r.parent_table)

load_levels, placed = [], set()
while len(placed) < len(pg_tables):                          # topological sort, level by level
    level = sorted(t for t in pg_tables if t not in placed and depends_on[t] <= placed)
    if not level:
        raise RuntimeError("circular foreign keys")
    load_levels.append(level)
    placed |= set(level)

pg_load_levels = pd.DataFrame({
    "level": range(1, len(load_levels) + 1),
    "tables": [", ".join(l) for l in load_levels],
    "depend_on_tables": ["-" if not any(depends_on[t] for t in l) else ", ".join(sorted(set().union(*[depends_on[t] for t in l]))) for l in load_levels],
}).set_index("level")
print("3.4 TABLE-LEVEL LINEAGE (a table can only be filled after the tables it references)")
display(pg_load_levels)
print("Parent -> children:")
for parent, kids in pg_fk.groupby("parent_table")["child_table"].agg(lambda x: sorted(set(x))).items():
    print(f"  {parent:<18} -> {', '.join(kids)}")
print("Tables nobody references:", ", ".join(t for t in pg_tables if t not in set(pg_fk["parent_table"])) or "-")

# ---- 3.5 Objects inside the database that transform or move data ----------------------------------------------------------
objects = pg_query("""
    select 'view' as kind, c.relname as name from pg_class c where c.relnamespace = 'public'::regnamespace and c.relkind in ('v','m')
    union all select 'function', p.proname from pg_proc p where p.pronamespace = 'public'::regnamespace
    union all select 'trigger', t.tgname from pg_trigger t join pg_class c on c.oid = t.tgrelid
              where c.relnamespace = 'public'::regnamespace and not t.tgisinternal
    union all select 'rule', r.rulename from pg_rules r where r.schemaname = 'public'
    union all select 'sequence', c.relname from pg_class c where c.relnamespace = 'public'::regnamespace and c.relkind = 'S'""")
print("\n3.5 VIEWS / FUNCTIONS / TRIGGERS / RULES / SEQUENCES IN SCHEMA public")
print("None found - no data is transformed inside the database." if objects.empty else objects.to_string(index=False))

# ---- 3.6 How rows were written: the database's own activity counters ---------------------------------------------------------
act = pg_query("""
    select relname as table_name, n_tup_ins as inserts, n_tup_upd as updates, n_tup_hot_upd as hot_updates,
           n_tup_del as deletes, n_live_tup as live_rows_estimate, n_dead_tup as dead_rows,
           last_vacuum, last_autovacuum, last_analyze, last_autoanalyze
    from pg_stat_user_tables order by relname""").set_index("table_name")
reset = pg_scalar("select stats_reset from pg_stat_database where datname = current_database()")
print(f"\n3.6 WRITE ACTIVITY SINCE THE STATISTICS WERE LAST RESET ({reset})")
display(act)

# ---- 3.7 Which rows changed after they were created, and when ------------------------------------------------------------------
chg = []
for t in pg_tables:
    cols = set(pg_columns.loc[pg_columns["table_name"] == t, "column_name"])
    if {"created_at", "updated_at"} <= cols:
        r = pg_query(sql.SQL("""select count(*) filter (where updated_at > created_at) as changed,
                                       min(updated_at) filter (where updated_at > created_at) as first_change,
                                       max(updated_at) as last_change from {t}""").format(t=ident(t))).iloc[0]
        chg.append((t, pg_rowcounts[t], int(r["changed"]), r["first_change"], r["last_change"]))
pg_change_lineage = pd.DataFrame(chg, columns=["table", "rows", "changed_after_creation", "first_change", "last_change"])
print("3.7 ROWS CHANGED AFTER CREATION (updated_at > created_at)")
display(pg_change_lineage)

# ---- 3.8 Derived columns discovered from the data: numeric column c ~ a x b in the same table --------------------------------
# Every numeric column c is tested against every product of two other numeric columns of the table.
found = []
for t in pg_tables:
    nums = pg_columns[(pg_columns["table_name"] == t) & (pg_columns["data_type"].isin(NUMERIC))]["column_name"].tolist()
    for c in nums:
        for i, a in enumerate(nums):
            for b in nums[i + 1:]:
                if c in (a, b):
                    continue
                r = pg_query(sql.SQL("""select count(*) filter (where abs({c}::numeric - {a}::numeric * {b}::numeric) < 0.01) as match_rows,
                                               count(*) as comparable
                                        from {t} where {c} is not null and {a} is not null and {b} is not null""")
                             .format(c=ident(c), a=ident(a), b=ident(b), t=ident(t))).iloc[0]
                if r["comparable"] and r["match_rows"] / r["comparable"] >= 0.5:
                    found.append((t, c, f"{a} x {b}", int(r["comparable"]), int(r["match_rows"]), int(r["comparable"] - r["match_rows"]),
                                  f"{r['match_rows'] / r['comparable'] * 100:.2f}%"))
pg_derived_columns = pd.DataFrame(found, columns=["table", "derived_column", "equals", "rows_compared", "match", "do_not_match", "match_%"])
print("3.8 DERIVED NUMERIC COLUMNS FOUND IN THE DATA (column = a x b, shown when >= 50% of rows agree)")
display(pg_derived_columns if len(pg_derived_columns) else pd.DataFrame({"result": ["none found"]}))

# ---- 3.9 Id columns that encode another column (number = prefix digits of a date) -------------------------------------------------
# For each single-column bigint/integer primary key: does its leading part equal the table's created_at date as YYYYMMDD?
enc = []
for t, c in pg_pk.groupby("table_name")["column_name"].agg(list).items():
    cols = set(pg_columns.loc[pg_columns["table_name"] == t, "column_name"])
    if len(c) == 1 and "created_at" in cols:
        r = pg_query(sql.SQL("""select count(*) filter (where left({k}::text, 8) = to_char(created_at, 'YYYYMMDD')) as match_rows, count(*) as n
                                from {t}""").format(k=ident(c[0]), t=ident(t))).iloc[0]
        if r["match_rows"] / r["n"] >= 0.5:
            enc.append((t, c[0], "first 8 digits = created_at as YYYYMMDD", int(r["n"]), int(r["match_rows"])))
print("3.9 IDS THAT CARRY A DATE")
display(pd.DataFrame(enc, columns=["table", "id_column", "observed_pattern", "rows", "match"]) if enc else pd.DataFrame({"result": ["none found"]}))

## 4. Data mapping
Foreign keys, same-name columns, business keys, raw-to-clean rules, links to leads / web logs (documented and discovered from the values), and the planned warehouse mapping.

In [ ]:
# =====================================================================
# POSTGRES SECTION 4 - DATA MAPPING  (how fields in one table correspond to fields in other tables)
# Everything below is derived from the database: declared foreign keys, columns that share a name and overlapping
# values, and raw-to-clean value variants.
# =====================================================================

# ---- 4.1 Declared foreign-key mapping: child column -> parent column, with an orphan test --------------------------------
fk_rows = []
for r in pg_fk.itertuples():
    q = sql.SQL("""select count(*) filter (where c.{cc} is null) as null_fk,
                          count(*) filter (where c.{cc} is not null and p.{pc} is null) as orphans,
                          count(*) as child_rows, count(distinct c.{cc}) as parents_used
                   from {ct} c left join {pt} p on p.{pc} = c.{cc}""").format(
        cc=ident(r.child_column), pc=ident(r.parent_column), ct=ident(r.child_table), pt=ident(r.parent_table))
    x = pg_query(q).iloc[0]
    parents = pg_rowcounts[r.parent_table]
    fk_rows.append((f"{r.child_table}.{r.child_column}", f"{r.parent_table}.{r.parent_column}", r.constraint_name,
                    int(x["child_rows"]), int(x["null_fk"]), int(x["orphans"]), int(x["parents_used"]), parents,
                    f"{x['parents_used'] / parents * 100:.1f}%", f"{x['child_rows'] / x['parents_used']:.1f}"))
pg_fk_mapping = pd.DataFrame(fk_rows, columns=["child_column", "parent_column", "constraint", "child_rows", "null_values",
                                               "orphans (no parent)", "parents_used", "parent_rows", "parents_used_%", "avg_children_per_used_parent"])
print("4.1 FOREIGN-KEY MAPPING (declared in the database, orphans re-tested)")
display(pg_fk_mapping)

# ---- 4.2 Columns that share a name across tables: do their values overlap? ----------------------------------------------------
# Shows relationships that exist in the data whether or not a foreign key was declared.
fk_pairs = {(r.child_table, r.child_column, r.parent_table, r.parent_column) for r in pg_fk.itertuples()}
fk_pairs |= {(b, d, a, c) for a, c, b, d in fk_pairs}
same = pg_columns[~pg_columns["data_type"].isin(DATETIME + ("boolean",))].groupby("column_name")["table_name"].agg(list)
rows = []
for col, tabs in same.items():
    for i, ta in enumerate(tabs):
        for tb in tabs[i + 1:]:
            r = pg_query(sql.SQL("""select (select count(distinct {c}) from {ta}) as da, (select count(distinct {c}) from {tb}) as db,
                                           (select count(*) from (select distinct {c} from {ta} where {c} is not null) x
                                            where {c}::text in (select {c}::text from {tb} where {c} is not null)) as common""")
                         .format(c=ident(col), ta=ident(ta), tb=ident(tb))).iloc[0]
            rows.append((col, ta, tb, int(r["da"]), int(r["db"]), int(r["common"]),
                         f"{r['common'] / r['da'] * 100:.1f}%" if r["da"] else "-", f"{r['common'] / r['db'] * 100:.1f}%" if r["db"] else "-",
                         "yes" if (ta, col, tb, col) in fk_pairs else "no"))
pg_same_name = pd.DataFrame(rows, columns=["column", "table_a", "table_b", "distinct_a", "distinct_b", "common_values",
                                           "common_of_a", "common_of_b", "declared_fk_between_these_columns"])
print("4.2 SAME-NAME COLUMNS IN DIFFERENT TABLES AND THEIR VALUE OVERLAP")
display(pg_same_name)

# ---- 4.3 Raw value -> clean value: spellings that differ only by case or surrounding spaces ----------------------------------
# For every text column: values that collapse into the same value after lower(trim()).
variants = []
for t, c in pg_columns[pg_columns["data_type"] == "text"][["table_name", "column_name"]].itertuples(index=False):
    d = pg_query(sql.SQL("""select lower(btrim({c})) as clean_value, array_agg(distinct {c}) as raw_spellings, count(*) as rows
                            from {t} where {c} is not null group by 1 having count(distinct {c}) > 1 order by 3 desc""")
                 .format(c=ident(c), t=ident(t)))
    if len(d):
        variants.append(dict(table=t, column=c, values_with_variants=len(d), rows_affected=int(d["rows"].sum()),
                             example_clean=d["clean_value"].iloc[0], example_raw=[repr(x) for x in d["raw_spellings"].iloc[0]]))
pg_value_variants = pd.DataFrame(variants)
print("4.3 TEXT COLUMNS WHOSE VALUES HAVE SEVERAL SPELLINGS (rule that unifies them: lower(trim(value)))")
display(pg_value_variants if len(pg_value_variants) else pd.DataFrame({"result": ["none found"]}))

In [ ]:
# ---- 4.4 Business-key mappings and raw -> clean value rules (tested in the database) ----------------------------------------------
BK = [
    ("order_items (supplier_id, product_id)", "supplier_products (supplier_id, product_id)", "COGS lookup: supplier price of the line",
     "select count(*) from order_items i left join supplier_products sp using (supplier_id, product_id) where sp.supplier_id is null"),
    ("suppliers.supplier_id", "companies.company_id", "supplier id equals the company id of the supplier",
     "select count(*) from suppliers where supplier_id <> company_id"),
    ("suppliers.company_id", "companies.company_type = 'supplier'", "a supplier must point to a supplier-type company",
     "select count(*) from suppliers s join companies c on c.company_id = s.company_id where c.company_type <> 'supplier'"),
    ("orders.buyer_company_id", "companies.company_type = 'buyer'", "orders must come from buyer-type companies",
     "select count(*) from orders o join companies c on c.company_id = o.buyer_company_id where c.company_type <> 'buyer'"),
    ("customers.company_id", "companies.company_type = 'buyer'", "customers work at buyer-type companies",
     "select count(*) from customers cu join companies c on c.company_id = cu.company_id where c.company_type <> 'buyer'"),
    ("orders.customer_id -> customers.company_id", "orders.buyer_company_id", "customer's employer = buying company",
     "select count(*) from orders o join customers c on c.customer_id = o.customer_id where c.company_id <> o.buyer_company_id"),
]
pg_business_key_mapping = pd.DataFrame(
    [(a, b, why, int(pg_scalar(q))) for a, b, why, q in BK],
    columns=["from", "to", "purpose", "rows_that_break_the_mapping"])
print("4.4 BUSINESS-KEY MAPPINGS (not enforced by constraints, tested here)")
display(pg_business_key_mapping)

# ---- 4.5 Raw value -> clean value (how dirty spellings map to the standard value) ------------------------------------
print("4.5 RAW VALUE -> CLEAN VALUE")
status_map = pg_query("""select status as raw_value, count(*) as rows,
                                lower(btrim(status)) as maps_to
                         from orders group by status order by 2 desc""")
status_map["is_clean"] = status_map["raw_value"] == status_map["maps_to"]
print("orders.status  (rule: lower(trim(status)))")
display(status_map)

gender_map = pg_query("""select coalesce(gender, '(NULL)') as raw_value, count(*) as rows,
                                case when upper(btrim(gender)) in ('M','MALE') then 'M'
                                     when upper(btrim(gender)) in ('F','FEMALE') then 'F' else 'unknown' end as maps_to
                         from customers group by gender order by 2 desc""")
print("customers.gender  (rule: M/Male -> M, F/Female -> F, everything else -> unknown)")
display(gender_map)

email_case = pg_query("""select count(*) filter (where email = lower(btrim(email))) as already_clean,
                                count(*) filter (where email <> lower(btrim(email))) as fixed_by_lower_trim
                         from customers""")
print("customers.email  (rule: lower(trim(email)) before any join)")
display(email_case)

country_map = pg_query("""select country as raw_value, count(*) as rows, 'same (full English name)' as maps_to
                          from companies group by country order by 2 desc""")
print("companies.country  (full name; the ISO-2 form is needed only when joining with web-log geo data)")
display(country_map)

In [ ]:
# ---- 4.6 Mapping to the OTHER sources (leads, web logs) ---------------------------------------------------------------
pg_cross_source_mapping = pd.DataFrame([
    ("customers.email", "Leads: contact_email (status = Won)", "lower(trim(a)) = lower(trim(b))", "PRIMARY: identifies the company a lead became"),
    ("companies.name", "Leads: company_name", "cleaned name (trim, case, dots) equal", "FALLBACK: typos break it"),
    ("companies.country", "Leads: country", "leads may use AR / UPPER / lower -> map to the full name", "conform before joining"),
    ("customers.email", "Web logs: auth_user", "lower(trim(a)) = lower(trim(b))", "who the logged-in visitor is"),
    ("orders.created_at (channel = web)", "Web logs: POST /checkout/confirm timestamp", "equal to the second, same customer", "which order a checkout hit created"),
    ("products.sku", "Web logs: /products/<sku> in the URL", "exact SKU", "product-level web interest"),
    ("companies.created_at", "Leads: created_date", "Won lead is created 5-25 days BEFORE the company", "lead -> first order lag"),
], columns=["postgres_field", "other_source.field", "mapping_rule", "purpose"])
print("4.6a CROSS-SOURCE MAPPING (documented)")
display(pg_cross_source_mapping)

# ---- 4.6b Test the leads mapping with real data (uses leads_raw if the leads notebook state exists, otherwise reads the files) -----------------------
if "leads_raw" not in globals():
    # This notebook is separate from the leads notebook: read just the 4 columns needed, straight from the Excel files
    # (read-only; the header row is found as the row that starts with 'lead_id').
    _parts = []
    for _f in sorted(Path(r"D:\dataropes\b2b_ecommerce_project\data\output\leads").glob("*.xlsx")):
        if _f.name.startswith("~$"):
            continue
        _s = pd.read_excel(_f, engine="openpyxl", header=None, dtype=object)
        _h = next((i for i in range(min(len(_s), 10)) if str(_s.iat[i, 0]).strip() == "lead_id"), None)
        if _h is None:
            continue
        _b = _s.iloc[_h + 1:].copy()
        _b.columns = [str(v).strip() for v in _s.iloc[_h]]
        _parts.append(_b.dropna(how="all"))
    leads_raw = pd.concat(_parts, ignore_index=True)
    print(f"(leads read for this test: {len(leads_raw):,} rows from {len(_parts)} files)")

if "leads_raw" in globals():
    won = leads_raw[leads_raw["status"].astype(str).str.strip().str.lower() == "won"].copy()
    won_email = won["contact_email"].dropna().astype(str).str.strip().str.lower()

    cust = pg_query("select customer_id, company_id, lower(btrim(email)) as email_clean from customers")
    comp = pg_query("select company_id, name from companies where company_type = 'buyer'")

    def clean_name(s):                                         # same idea as the docs: trim + case + dots
        return re.sub(r"\s+", " ", str(s).replace(".", "")).strip().lower()

    comp_names = {clean_name(n) for n in comp["name"]}
    won_distinct = won.drop_duplicates("lead_id")
    by_email = won_distinct["contact_email"].astype(str).str.strip().str.lower().isin(set(cust["email_clean"]))
    by_name = won_distinct["company_name"].map(clean_name).isin(comp_names)

    mapped_customers = cust[cust["email_clean"].isin(set(won_email))]
    pg_leads_match = pd.Series({
        "Won leads (distinct lead_id)": len(won_distinct),
        "... matched to a customer by e-mail": int(by_email.sum()),
        "... matched to a buyer company by cleaned name": int(by_name.sum()),
        "... matched by e-mail OR name": int((by_email | by_name).sum()),
        "... matched by neither": int((~(by_email | by_name)).sum()),
        "Distinct companies reached through the e-mail link": mapped_customers["company_id"].nunique(),
    })
    print("4.6b LIVE TEST: leads -> PostgreSQL")
    print(pg_leads_match.to_string())
else:
    print("4.4b skipped: run the leads sections first (leads_raw is needed for the live match test).")

In [ ]:
# ---- 4.7 Mapping between the database and the leads Excel files (connection discovered from the values) -------------------
# The leads files are only read. Every leads column is compared with every text / integer column of the database:
# the share of the leads' distinct values (lower-cased, trimmed) that also exist in that database column.
LEADS_DIR = Path(r"D:\dataropes\b2b_ecommerce_project\data\output\leads")

if "leads_raw" not in globals():
    parts = []
    for f in sorted(LEADS_DIR.glob("*.xlsx")):
        if f.name.startswith("~$"):
            continue
        s = pd.read_excel(f, engine="openpyxl", header=None, dtype=object)
        top = s.head(10)
        counts = top.notna().sum(axis=1)
        h = int(counts[counts == counts.max()].index[0])             # header row = first row with the most filled cells
        body = s.iloc[h + 1:].copy()
        body.columns = [str(v).strip() for v in s.iloc[h]]
        parts.append(body.dropna(how="all"))
    leads_raw = pd.concat(parts, ignore_index=True)
    print(f"(leads read for this comparison: {len(leads_raw):,} rows from {len(parts)} files)")


def norm(series):
    return series.dropna().astype(str).str.strip().str.lower().loc[lambda s: s != ""]


# database side: distinct normalised values per text / integer column
db_values = {}
for t, c, dt in pg_columns[pg_columns["data_type"] == "text"][["table_name", "column_name", "data_type"]].itertuples(index=False):   # text columns only: matching bare numbers would be coincidence
    v = pg_query(sql.SQL("select distinct lower(btrim({c}::text)) as v from {t} where {c} is not null").format(c=ident(c), t=ident(t)))["v"]
    db_values[(t, c)] = set(v[v != ""])

rows = []
for lc in [c for c in leads_raw.columns if not str(c).startswith("_")]:
    vals = norm(leads_raw[lc])
    if vals.nunique() < 2 or pd.to_numeric(vals.str.replace(r"[$,]", "", regex=True), errors="coerce").notna().mean() > 0.9:
        continue                                          # skip numeric leads columns (a number matching a number is coincidence)
    lv = set(vals)
    for (t, c), dv in db_values.items():
        common = len(lv & dv)
        if common:
            rows.append((lc, f"{t}.{c}", len(lv), len(dv), common, round(common / len(lv) * 100, 1), round(common / len(dv) * 100, 1)))
pg_leads_overlap = (pd.DataFrame(rows, columns=["leads_column", "database_column", "leads_distinct", "db_distinct", "common_values",
                                                "common_of_leads_%", "common_of_db_%"])
                    .sort_values(["common_values", "common_of_leads_%"], ascending=False).reset_index(drop=True))
print("4.7 LEADS COLUMN <-> DATABASE COLUMN VALUE OVERLAP (pairs with at least 1 common value, strongest first)")
display(pg_leads_overlap.head(30))

In [ ]:
# ---- 4.8 Row-level connection: which leads rows find a partner row in the database, and what they have in common -------------
# Uses the strongest "identifier-like" pair from 4.4 (leads column with many distinct values and a clear overlap).
candidates = pg_leads_overlap[(pg_leads_overlap["leads_distinct"] >= 100) & (pg_leads_overlap["common_values"] >= 20)]
if candidates.empty:
    print("No leads column with many distinct values overlaps a database column - no row-level connection found.")
else:
    for _, p in candidates.head(3).iterrows():
        lcol, dbcol = p["leads_column"], p["database_column"]
        t, c = dbcol.split(".")
        keys = db_values[(t, c)]
        matched = leads_raw[lcol].astype(str).str.strip().str.lower().isin(keys) & leads_raw[lcol].notna()
        print(f"4.8 {lcol}  ->  {dbcol}")
        print(f"    leads rows: {len(leads_raw):,} | rows whose value exists in {dbcol}: {int(matched.sum()):,} ({matched.mean() * 100:.1f}%)")

        # what do the matched rows have in common? (every low-cardinality leads column)
        for oc in leads_raw.columns:
            if str(oc).startswith("_") or oc == lcol:
                continue
            vals = leads_raw[oc].dropna().astype(str).str.strip()
            if 2 <= vals.nunique() <= 12:
                tab = pd.crosstab(leads_raw.loc[vals.index, oc].astype(str).str.strip(), matched.loc[vals.index].map({True: "matched", False: "not matched"}))
                tab["matched_%"] = (tab.get("matched", 0) / tab.sum(axis=1) * 100).round(1)
                if tab["matched_%"].max() - tab["matched_%"].min() >= 50:          # only show columns that separate matched rows
                    print(f"\n    Matched share by leads column '{oc}':")
                    display(tab)
        print()

In [ ]:
# ---- 4.9 Mapping source -> staging -> warehouse (PLANNED design from docs section 11, requirements.md) -------------------
# Nothing here exists yet in the database; it is the target the pipeline is meant to build.
pg_warehouse_mapping = pd.DataFrame([
    ("companies", "stg_pg_companies", "dim_company (SCD2, buyers) + dim_supplier attributes (suppliers)", "split by company_type; keep version history; ISO-2 country; clean CUIT"),
    ("suppliers", "stg_pg_suppliers", "dim_supplier", "join companies for name / country; lead_time_days, is_preferred"),
    ("products", "stg_pg_products", "dim_product", "sku, name, category, brand, list_price"),
    ("supplier_products", "stg_pg_supplier_products", "fact_order_item.supplier_price (COGS) / supplier-product KPI", "one static price per pair"),
    ("customers", "stg_pg_customers", "dim_customer", "email_clean = lower(trim(email)); gender_clean; keep company link"),
    ("orders", "stg_pg_orders", "fact_order (optional) + status / channel on fact_order_item", "status = lower(trim(status)); order_date repaired from created_at; rolling-window CDC"),
    ("order_items", "stg_pg_order_items", "fact_order_item (partitioned by order date)", "dedupe (order_id, product_id); repair quantity / unit_price from line_total; revenue = line_total"),
], columns=["postgres_table", "staging_table (planned)", "warehouse_target (planned)", "key_transformations"]).set_index("postgres_table")

# which source columns appear in the dictionary as having rule violations (= must be cleaned on the way)
needs_cleaning = (pg_data_dictionary[pg_data_dictionary["rule_violations"] > 0]
                  .reset_index().groupby("table")["column"].agg(lambda x: ", ".join(x)))
pg_warehouse_mapping["columns_that_need_cleaning (measured)"] = needs_cleaning
pg_warehouse_mapping["columns_that_need_cleaning (measured)"] = pg_warehouse_mapping["columns_that_need_cleaning (measured)"].fillna("-")

print("4.9 SOURCE -> STAGING -> WAREHOUSE (planned)")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 250)
pg_warehouse_mapping

## 5. Business glossary
Plain-language definitions with the live number from the database, plus the vocabulary found in the database itself.

In [ ]:
# =====================================================================
# POSTGRES SECTION 5 - BUSINESS GLOSSARY  (plain-language meaning of each term, with the live number from the database)
# Definitions come from the project docs and generator. Numbers are calculated with read-only SELECTs.
# Orders with status NULL are treated as not cancelled; status is compared as lower(trim(status)).
# =====================================================================

# ---- 5.1 Numbers used by the glossary ------------------------------------------------------------------------------------
n = pg_query("""
    select
      (select count(*) from companies where company_type = 'buyer')                               as buyers,
      (select count(*) from companies where company_type = 'supplier')                            as suppliers,
      (select count(*) from companies where company_type = 'buyer' and is_active)                 as active_buyers,
      (select count(*) from customers)                                                            as customers,
      (select count(*) from customers where is_active)                                            as active_customers,
      (select count(*) from orders)                                                               as orders,
      (select count(*) from orders where lower(btrim(status)) = 'cancelled')                      as cancelled,
      (select count(*) from order_items)                                                          as lines,
      (select min(created_at)::date from orders)                                                  as first_order,
      (select max(created_at)::date from orders)                                                  as last_order,
      (select count(*) from products)                                                             as products,
      (select count(distinct category) from products)                                             as categories,
      (select count(*) from products where not is_active)                                         as inactive_products,
      (select round(100.0 * count(*) filter (where is_preferred) / count(*), 1) from suppliers)   as preferred_suppliers_pct,
      (select round(avg(lead_time_days), 1) from suppliers)                                       as avg_supplier_lead,
      (select count(*) from companies where cuit ~ '^\\d{2}-\\d{8}-\\d$')                         as cuit_ok
""").iloc[0]

revenue = pg_query("""
    with lines as (
        select i.*, lower(btrim(o.status)) as st,
               row_number() over (partition by i.order_id, i.product_id order by i.order_item_id) as rn
        from order_items i join orders o on o.order_id = i.order_id)
    select sum(line_total)                                                         as gross_all,
           sum(line_total) filter (where st is distinct from 'cancelled')          as net,
           sum(line_total) filter (where st is distinct from 'cancelled' and rn = 1) as net_deduped,
           count(distinct order_id) filter (where st is distinct from 'cancelled') as orders_net
    from lines""").iloc[0]

margin = pg_query("""
    select sum(i.line_total) as revenue, sum(i.quantity * sp.supplier_price) as cogs
    from order_items i
    join orders o on o.order_id = i.order_id
    join supplier_products sp on sp.supplier_id = i.supplier_id and sp.product_id = i.product_id
    where lower(btrim(o.status)) is distinct from 'cancelled' and i.quantity > 0 and i.unit_price > 0""").iloc[0]

price_ratio = pg_query("""
    select round(avg(i.unit_price / p.list_price), 3) as unit_vs_list,
           round(avg(sp.supplier_price / p.list_price), 3) as cost_vs_list,
           round(avg(i.discount_pct), 2) as avg_discount
    from order_items i join products p on p.product_id = i.product_id
    join supplier_products sp on sp.supplier_id = i.supplier_id and sp.product_id = i.product_id
    where i.unit_price > 0""").iloc[0]

by_buyer = pg_query("""
    select o.buyer_company_id, sum(i.line_total) as net, count(distinct o.order_id) as orders
    from orders o join order_items i on i.order_id = o.order_id
    where lower(btrim(o.status)) is distinct from 'cancelled'
    group by 1 order by 2 desc""")
top20 = by_buyer.head(max(1, round(len(by_buyer) * 0.2)))["net"].sum() / by_buyer["net"].sum() * 100
one_time = int((by_buyer["orders"] == 1).sum())


def dist(query):
    d = pg_query(query)
    return ", ".join(f"{a} {b}" for a, b in zip(d.iloc[:, 0], d.iloc[:, 1]))


status_dist = dist("select lower(btrim(status)) as s, count(*) from orders group by 1 order by 2 desc")
channel_dist = dist("select channel, count(*) from orders group by 1 order by 2 desc")
payment_dist = dist("select payment_method, count(*) from orders group by 1 order by 2 desc")
ship_dist = dist("select shipping_method, count(*) from orders group by 1 order by 2 desc")
tax_dist = dist("select tax_rate, count(*) from order_items group by 1 order by 2 desc")
wh_dist = dist("select warehouse, count(*) from order_items group by 1 order by 2 desc")
cat_dist = dist("select category, count(*) from products group by 1 order by 1")

money = lambda v: f"{float(v):,.0f}"

# ---- 5.2 Glossary --------------------------------------------------------------------------------------------------------------
pg_glossary = pd.DataFrame([
    ("Buyer company", "A company that purchases from the marketplace", "companies.company_type = 'buyer'", f"{n.buyers:,} buyers ({n.active_buyers:,} active)"),
    ("Supplier", "A company that supplies products to the marketplace; has a row in companies AND suppliers with the same id", "companies.company_type = 'supplier'; suppliers", f"{n.suppliers:,} suppliers; {n.preferred_suppliers_pct}% preferred; avg lead time {n.avg_supplier_lead} days"),
    ("Customer", "A person at a buyer company who places orders and logs in to the web shop", "customers", f"{n.customers:,} customers ({n.active_customers:,} active)"),
    ("Order", "One purchase by a buyer; the header of one or more order lines", "orders (1 row = 1 order)", f"{n.orders:,} orders from {n.first_order} to {n.last_order}"),
    ("Order line", "One product on one order; the grain of all revenue numbers", "order_items (1 row = 1 line)", f"{n.lines:,} lines, {n.lines / n.orders:.1f} per order"),
    ("Order status / lifecycle", "Where the order is: pending > confirmed > shipped > delivered, or cancelled. Changes in place over time", "orders.status (clean with lower(trim()))", status_dist),
    ("Cancelled order / cancellation rate", "An order that was cancelled; excluded from net revenue", "status = 'cancelled'", f"{n.cancelled:,} orders = {n.cancelled / n.orders * 100:.1f}%"),
    ("Order channel", "Where the order originated; only 'web' orders leave a trace in the web logs", "orders.channel", channel_dist),
    ("Payment method", "How the buyer pays", "orders.payment_method", payment_dist),
    ("Shipping method", "How the order is delivered; shipping cost is NOT part of revenue", "orders.shipping_method", ship_dist),
    ("PO number", "The buyer's own purchase-order reference; legitimately empty on some orders", "orders.po_number", f"empty on {int(pg_scalar('select count(*) from orders where po_number is null')):,} orders"),
    ("Gross revenue", "Sum of line_total over all orders, cancelled included", "sum(order_items.line_total)", money(revenue.gross_all)),
    ("Net revenue", "Merchandise revenue excluding cancelled orders, tax and shipping (duplicate lines still included)", "sum(line_total) where status <> cancelled", money(revenue.net)),
    ("Net revenue (deduplicated)", "Net revenue after removing duplicate (order, product) lines - the figure to trust", "line_total, rn = 1 per (order_id, product_id)", f"{money(revenue.net_deduped)}  (duplicates add {revenue.net - revenue.net_deduped:,.0f})"),
    ("Average order value (AOV)", "Net revenue divided by the number of non-cancelled orders", "net revenue / orders", f"{revenue.net_deduped / revenue.orders_net:,.0f}"),
    ("List price", "Catalog price of a product", "products.list_price", f"{n.products:,} products in {n.categories} categories: {cat_dist}"),
    ("Unit price", "Price actually charged per unit, already net of discount; can be a little above list price", "order_items.unit_price", f"on average {price_ratio.unit_vs_list * 100:.1f}% of list price"),
    ("Supplier price (cost proxy / COGS)", "What the supplier charges; used as the cost of goods because no cost column exists", "supplier_products.supplier_price", f"on average {price_ratio.cost_vs_list * 100:.1f}% of list price"),
    ("Discount", "Informational discount; it is already inside unit_price, never subtract it again", "order_items.discount_pct", f"average {price_ratio.avg_discount}%"),
    ("Tax rate", "Argentine VAT rate of the line; NOT included in line_total", "order_items.tax_rate", tax_dist),
    ("Gross margin (merchandise margin)", "Revenue minus supplier cost on the same lines, divided by revenue (clean, non-cancelled lines only; approximate)", "(line_total - qty x supplier_price) / line_total",
     f"{(margin.revenue - margin.cogs) / margin.revenue * 100:.1f}%"),
    ("Pareto / top-20% buyers", "Share of net revenue that comes from the top 20% of buyers that ordered", "net revenue by buyer_company_id", f"{top20:.1f}% of net revenue from the top 20% of {len(by_buyer):,} buying companies"),
    ("One-time buyer", "A buyer company with exactly one non-cancelled order", "orders per buyer_company_id", f"{one_time:,} of {len(by_buyer):,} buyers ({one_time / len(by_buyer) * 100:.1f}%)"),
    ("CUIT", "Argentine company tax id formatted NN-NNNNNNNN-N", "companies.cuit", f"{n.cuit_ok:,} of {n.buyers + n.suppliers:,} companies have the clean format"),
    ("SKU", "Product code, also visible in web-log URLs /products/<sku>", "products.sku", "e.g. " + ", ".join(pg_query("select sku from products order by product_id limit 3")["sku"])),
    ("Preferred supplier", "Flag for a favoured supplier (at supplier level and, independently, at supplier-product level)", "suppliers.is_preferred, supplier_products.is_preferred", f"{n.preferred_suppliers_pct}% of suppliers"),
    ("Lead time", "Days a supplier needs to deliver (supplier level and pair level differ - do not mix)", "suppliers.lead_time_days, supplier_products.lead_time_days", f"supplier average {n.avg_supplier_lead} days"),
    ("Minimum order quantity / value", "Supplier / product ordering minimums - recorded but NOT enforced by the data", "products.min_order_qty, suppliers.min_order_value", "not enforced"),
    ("Warehouse", "Dispatch site of an order line; random and unrelated to the buyer's geography", "order_items.warehouse", wh_dist),
    ("Active / inactive", "Flag on companies, customers and products; inactive products still sell and inactive customers still order", "is_active", f"{n.inactive_products:,} inactive products, {n.customers - n.active_customers:,} inactive customers"),
    ("Won lead", "A marketing lead that became a buyer company: links the leads dataset to customers via e-mail", "customers.email = leads.contact_email", "see section 4.4b"),
], columns=["term", "business_definition", "how_it_is_stored", "in_this_data"]).set_index("term")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)
pg_glossary

In [ ]:
# ---- 5.3 Vocabulary found in the database itself (added to the definitions above) --------------------------------------
dd = pg_data_dictionary.reset_index()
# ---- 5.3 Terms that appear in several tables (same column name) -------------------------------------------------------------------
shared = pg_columns.groupby("column_name")["table_name"].agg(lambda x: sorted(set(x)))
shared = shared[shared.map(len) > 1]
print("5.3 TERMS USED IN MORE THAN ONE TABLE")
display(pd.DataFrame({"term": shared.index, "used_in_tables": shared.values}).reset_index(drop=True))

# ---- 5.4 Controlled vocabulary: every value of every category column with its row count ----------------------------------
vocab = []
for r in dd[dd["role"] == "category"].itertuples():
    for value, n in r.allowed_values_found.items():
        vocab.append((r.table, r.column, value, n, f"{n / pg_rowcounts[r.table] * 100:.1f}%"))
pg_vocabulary = pd.DataFrame(vocab, columns=["table", "column", "value", "rows", "share"])
print("5.4 CONTROLLED VOCABULARY (values of category columns)")
display(pg_vocabulary)

## 6. Data catalog
Searchable inventory of tables, columns and relationships.

In [ ]:
# =====================================================================
# POSTGRES SECTION 6 - DATA CATALOG  (searchable inventory of the tables, columns and relationships)
# Combines the metadata, dictionary, lineage, mapping and glossary built above.
# =====================================================================

# ---- 6.1 Table catalog ---------------------------------------------------------------------------------------------------------------
TABLE_INFO = {   # description, grain, KPIs fed (docs section 2 / 9), tags
    "companies": ("Suppliers and buyer companies with location, tax id, rating", "1 row = 1 company", "K2 K5 K6 K7 K13", "company, buyer, supplier, country, cuit"),
    "suppliers": ("Supplier-specific attributes (lead time, minimums, preferred flag)", "1 row = 1 supplier", "K13 K14", "supplier, lead time"),
    "products": ("Product catalog with SKU, category and list price", "1 row = 1 product", "K3 K4 K14", "product, sku, category, price"),
    "supplier_products": ("Which supplier can supply which product, at what price (cost proxy)", "1 row = 1 (supplier, product) pair", "K4 K14", "supplier price, cogs, margin"),
    "customers": ("People at buyer companies who place orders; e-mail is the cross-source key", "1 row = 1 person", "K5 K6 + web / leads bridge", "customer, email, contact"),
    "orders": ("Order header: status, channel, payment, shipping, dates", "1 row = 1 order", "K1 K2 K5 K6 K7 K8 K11 K12 K13", "order, status, channel, revenue"),
    "order_items": ("Order lines: product, supplier, quantity, price, total (the fact grain)", "1 row = 1 product on 1 order", "K1-K4 K7 K13 K14", "order line, revenue, quantity, price"),
}

dirty_by_table = pg_data_dictionary.reset_index().groupby("table")["rule_violations"].sum()
pg_catalog = pd.DataFrame({
    "description": {t: TABLE_INFO[t][0] for t in pg_tables},
    "grain": {t: TABLE_INFO[t][1] for t in pg_tables},
    "primary_key": pg_table_meta["primary_key"],
    "rows": pg_table_meta["rows"],
    "columns": pg_table_meta["columns"],
    "size_kb": pg_table_meta["total_size_kb"],
    "references": pg_table_meta["fk_out (references)"],
    "referenced_by": pg_table_meta["fk_in (referenced by)"],
    "rule_violations (sum over columns)": dirty_by_table,
    "kpis_fed": {t: TABLE_INFO[t][2] for t in pg_tables},
    "tags": {t: TABLE_INFO[t][3] for t in pg_tables},
    "location": f"{PG_SETTINGS['host']}:{PG_SETTINGS['port']}/{PG_SETTINGS['dbname']}.public",
    "refresh": {t: ("Daily inserts + status updates" if t in ("orders", "order_items") else "Loaded once (reference data)") for t in pg_tables},
})
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
print("6.1 TABLE CATALOG")
display(pg_catalog)

# ---- 6.2 Column catalog with quality flag ------------------------------------------------------------------------------------
pg_column_catalog = pg_data_dictionary.reset_index()[["table", "column", "type", "key", "meaning", "null_%", "distinct", "rule_violations", "findings"]].copy()
pg_column_catalog["quality"] = pg_column_catalog.apply(
    lambda r: "NEEDS CLEANING" if r["rule_violations"] > 0 else ("HAS FINDINGS" if r["findings"] != "-" else "OK"), axis=1)
pg_column_catalog["glossary_terms"] = pg_column_catalog.apply(
    lambda r: ", ".join(t for t, g in pg_glossary.iterrows() if f"{r['table']}.{r['column']}" in g["how_it_is_stored"]
                        or re.search(rf"\b{re.escape(r['column'])}\b", g["how_it_is_stored"]) and r["table"] in g["how_it_is_stored"]), axis=1)
print("6.2 COLUMN CATALOG (quality flag per column)")
print(pg_column_catalog["quality"].value_counts().to_string(), "\n")
display(pg_column_catalog)

# ---- 6.3 Relationship catalog ------------------------------------------------------------------------------------------------
print("6.3 RELATIONSHIP CATALOG (foreign keys + cross-source links)")
display(pg_fk_mapping[["child_column", "parent_column", "child_rows", "orphans (no parent)"]])


# ---- 6.4 Search the catalog ------------------------------------------------------------------------------------------------------
def pg_search_catalog(keyword):
    """Case-insensitive search over tables, columns and glossary terms. Example: pg_search_catalog('price')."""
    k = keyword.lower()
    return {
        "tables": pg_catalog[pg_catalog.apply(lambda r: k in (r.name + " " + " ".join(map(str, r.values))).lower(), axis=1)].index.tolist(),
        "columns": [f"{r.table}.{r.column}" for r in pg_column_catalog.itertuples()
                    if k in f"{r.table} {r.column} {r.meaning} {r.key}".lower()],
        "values in category columns": [f"{r.table}.{r.column} = {r.value}" for r in pg_vocabulary.itertuples() if k in str(r.value).lower()],
        "glossary terms": pg_glossary[pg_glossary.apply(lambda r: k in (r.name + " " + " ".join(map(str, r.values))).lower(), axis=1)].index.tolist(),
    }


# Example: change the keyword to search for anything
pg_search_catalog("price")